# Кольцевые коммуникации

Реализуйте три операции на CPU с Gloo. 
- `rank` — номер процесса,
- `world_size` — число процессов.
- Входные тензоры — одномерные CPU-тензоры одинакового размера на всех процессах. Их менять нельзя

In [1]:
%%writefile collectives.py
"""Ring SUM collectives on CPU/Gloo; no input mutations or native collectives."""
import torch
import torch.distributed as dist


def _exchange(outgoing, incoming, next_rank, previous_rank):
    pending = dist.batch_isend_irecv([
        dist.P2POp(dist.isend, outgoing, next_rank),
        dist.P2POp(dist.irecv, incoming, previous_rank),
    ])
    for request in pending:
        request.wait()


def ring_reduce_scatter(tensor):
    rank, size = dist.get_rank(), dist.get_world_size()
    if tensor.ndim != 1 or tensor.numel() == 0 or tensor.numel() % size:
        raise ValueError("Expected a nonempty 1D tensor divisible by world_size")
    chunks = [part.clone().contiguous() for part in tensor.chunk(size)]
    for step in range(size - 1):
        send_index = (rank - step - 1) % size
        receive_index = (rank - step - 2) % size
        received = torch.empty_like(chunks[receive_index])
        _exchange(chunks[send_index], received, (rank + 1) % size, (rank - 1) % size)
        chunks[receive_index].add_(received)
    return chunks[rank]


def ring_all_gather(shard):
    rank, size = dist.get_rank(), dist.get_world_size()
    if shard.ndim != 1 or shard.numel() == 0:
        raise ValueError("Expected a nonempty 1D shard")
    chunks = [torch.empty_like(shard) for _ in range(size)]
    chunks[rank].copy_(shard)
    for step in range(size - 1):
        send_index = (rank - step) % size
        receive_index = (rank - step - 1) % size
        _exchange(chunks[send_index], chunks[receive_index], (rank + 1) % size, (rank - 1) % size)
    return torch.cat(chunks)


def ring_all_reduce(tensor):
    return ring_all_gather(ring_reduce_scatter(tensor))


Overwriting collectives.py


## Сравнение с PyTorch

Сравним нашу реализацию с готовыми функциями торча `dist.reduce_scatter_tensor`, `dist.all_gather`
и `dist.all_reduce` на нескольких размерах тензоров. 

In [2]:
%%writefile check.py
from datetime import timedelta

import torch
import torch.distributed as dist

from collectives import ring_all_gather, ring_all_reduce, ring_reduce_scatter


torch.set_num_threads(1)
dist.init_process_group("gloo", timeout=timedelta(seconds=30))
try:
    rank = dist.get_rank()
    world_size = dist.get_world_size()
    generator = torch.Generator().manual_seed(42 + rank)

    for chunk_size in (1, 7, 31):
        values = torch.randn(world_size * chunk_size, generator=generator, dtype=torch.float64)
        original = values.clone()
        expected_shard = torch.empty(chunk_size, dtype=values.dtype)
        dist.reduce_scatter_tensor(expected_shard, values.clone(), op=dist.ReduceOp.SUM)
        torch.testing.assert_close(ring_reduce_scatter(values), expected_shard)
        torch.testing.assert_close(values, original, rtol=0, atol=0)

        shard = torch.randn(chunk_size, generator=generator, dtype=torch.float64)
        original_shard = shard.clone()
        expected_parts = [torch.empty_like(shard) for _ in range(world_size)]
        dist.all_gather(expected_parts, shard)
        torch.testing.assert_close(ring_all_gather(shard), torch.cat(expected_parts))
        torch.testing.assert_close(shard, original_shard, rtol=0, atol=0)

        expected_sum = values.clone()
        dist.all_reduce(expected_sum, op=dist.ReduceOp.SUM)
        torch.testing.assert_close(ring_all_reduce(values), expected_sum)
        torch.testing.assert_close(values, original, rtol=0, atol=0)

    if rank == 0:
        print(f"OK: все три операции прошли проверки на {world_size} процессах.")
finally:
    dist.destroy_process_group()

Overwriting check.py


In [3]:
import sys, subprocess
subprocess.run([sys.executable, '-m', 'torch.distributed.run', '--standalone', '--nproc-per-node=2', 'check.py'], check=True)

OK: все три операции прошли проверки на 2 процессах.


CompletedProcess(args=['/home/edu/venv-hw2/bin/python3', '-m', 'torch.distributed.run', '--standalone', '--nproc-per-node=2', 'check.py'], returncode=0)

In [4]:
import sys, subprocess
subprocess.run([sys.executable, '-m', 'torch.distributed.run', '--standalone', '--nproc-per-node=4', 'check.py'], check=True)

OK: все три операции прошли проверки на 4 процессах.


CompletedProcess(args=['/home/edu/venv-hw2/bin/python3', '-m', 'torch.distributed.run', '--standalone', '--nproc-per-node=4', 'check.py'], returncode=0)

## Дополнительные проверки: 1 и 3 процесса

Те же проверки курса: SUM, порядок шардов и неизменность входных тензоров.

In [5]:
for size in (1, 3):
    subprocess.run([sys.executable, '-m', 'torch.distributed.run', '--standalone', f'--nproc-per-node={size}', 'check.py'], check=True)

OK: все три операции прошли проверки на 1 процессах.


OK: все три операции прошли проверки на 3 процессах.
